# Segunda entrega - Trabalho de KNN

**Dataset:** `phishing_dataset.csv`  
**Objetivo:** usar o algoritmo KNN para classificar o valor da coluna `status`.

Integrantes do grupo:
- Orivaldo Eder Matavelli - RA: 52624665
- Max Guzman - RA: 52624436
- Kayky Duarte Del Bosque - RA: 52624973
- Pedro Donatz de Camargo Silva - RA: 52625361
- Heitor Samuel de Oliveira Barros - RA: 


## Notebook 1 - KNN feito manualmente

Neste notebook a ideia foi fazer o KNN sem usar o `KNeighborsClassifier`. A gente usa bibliotecas só para carregar dados, organizar a tabela e fazer contas básicas.

## 1. Importando as bibliotecas

Aqui foram usadas bibliotecas simples. O algoritmo KNN em si vai ser programado nas funções mais abaixo.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

## 2. Carregando o dataset

Primeiro vamos carregar a base de dados que foi usada no trabalho.

In [2]:
dados = pd.read_csv('phishing_dataset.csv')

dados.head()

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


## 3. Olhando informações básicas da base

Essa parte ajuda a entender o tamanho da base, as colunas e se existe algum valor vazio.

In [3]:
print('Quantidade de linhas e colunas:', dados.shape)
print('\nColunas da base:')
print(dados.columns.tolist())
print('\nValores vazios por coluna:')
print(dados.isnull().sum())

Quantidade de linhas e colunas: (11430, 89)

Colunas da base:
['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_external_redirection', 'length_words_raw', 'char_repeat', 'shortest_words_raw', 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 'longest_word_host', 'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path', 'phish_hints', 'domain_in_brand', 'brand_in_subdomain', 'brand_in_path', 'suspecious_tld', 'statistical_report', 'nb_hyperlinks', 'ratio_intHyperlinks', 'r

## 4. Definindo entrada e saída

A coluna `status` é a classe que queremos prever. A coluna `url` foi retirada porque é texto e não entra direto no cálculo da distância euclidiana.

In [4]:
y = dados['status']
X = dados.drop(['status', 'url'], axis=1)

print('Formato de X:', X.shape)
print('Formato de y:', y.shape)
print('Classes encontradas:')
print(y.value_counts())

Formato de X: (11430, 87)
Formato de y: (11430,)
Classes encontradas:
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


## 5. Separando treino e teste sem usar função pronta

Para deixar o processo mais manual, foi criada uma divisão simples com 80% dos dados para treino e 20% para teste.

In [5]:
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

tamanho_teste = int(len(X) * 0.2)
indices_teste = indices[:tamanho_teste]
indices_treino = indices[tamanho_teste:]

X_treino = X.iloc[indices_treino].to_numpy()
X_teste = X.iloc[indices_teste].to_numpy()
y_treino = y.iloc[indices_treino].to_numpy()
y_teste = y.iloc[indices_teste].to_numpy()

print('Treino:', X_treino.shape)
print('Teste:', X_teste.shape)

Treino: (9144, 87)
Teste: (2286, 87)


## 6. Função da distância euclidiana

O KNN compara um exemplo novo com os exemplos que já estão no treino. Para isso, usamos a distância euclidiana.

In [6]:
def distancia_euclidiana(a, b):
    soma = 0
    for i in range(len(a)):
        soma += (a[i] - b[i]) ** 2
    return soma ** 0.5

## 7. Função para prever uma linha

A função abaixo calcula a distância da linha de teste para todas as linhas de treino, ordena essas distâncias e pega os K vizinhos mais próximos.

In [7]:
def prever_uma_linha(linha_teste, X_treino, y_treino, k):
    distancias = []

    for i in range(len(X_treino)):
        d = distancia_euclidiana(linha_teste, X_treino[i])
        distancias.append((d, y_treino[i]))

    distancias.sort(key=lambda item: item[0])
    vizinhos = distancias[:k]

    classes = []
    for distancia, classe in vizinhos:
        classes.append(classe)

    voto = Counter(classes).most_common(1)[0][0]
    return voto

## 8. Função KNN manual

Agora aplicamos a previsão para todas as linhas de teste.

In [8]:
def knn_manual(X_treino, y_treino, X_teste, k):
    previsoes = []

    for linha in X_teste:
        resultado = prever_uma_linha(linha, X_treino, y_treino, k)
        previsoes.append(resultado)

    return np.array(previsoes)

## 9. Testando com K = 3

O valor de K escolhido inicialmente foi 3, igual ao que usamos no notebook com biblioteca.

In [ ]:
k = 3
previsoes = knn_manual(X_treino, y_treino, X_teste, k)

acertos = np.sum(previsoes == y_teste)
acuracia = acertos / len(y_teste)

print('K usado:', k)
print('Acertos:', acertos)
print('Total de testes:', len(y_teste))
print('Acurácia:', acuracia)

## 10. Testando valores diferentes de K

Também testamos alguns valores de K para ver se o resultado muda muito.

In [ ]:
resultados = []

for k in [1, 3, 5, 7, 9]:
    previsoes_k = knn_manual(X_treino, y_treino, X_teste, k)
    acuracia_k = np.mean(previsoes_k == y_teste)
    resultados.append([k, acuracia_k])

resultado_manual = pd.DataFrame(resultados, columns=['K', 'Acuracia'])
resultado_manual

## 11. Matriz de confusão simples

A matriz abaixo mostra onde o modelo acertou e errou para cada classe.

In [ ]:
tabela_confusao = pd.crosstab(
    pd.Series(y_teste, name='Real'),
    pd.Series(previsoes, name='Previsto')
)

tabela_confusao

## 12. Conclusão do notebook manual

Com o KNN manual foi possível entender melhor o passo a passo do algoritmo: calcular as distâncias, ordenar os vizinhos, pegar os mais próximos e escolher a classe por votação.

A parte mais trabalhosa foi montar as funções manualmente, principalmente porque o algoritmo precisa comparar cada linha de teste com todas as linhas de treino. Mesmo assim, o código ajudou a visualizar melhor a lógica do método.